<a href="https://colab.research.google.com/github/osergioribeirof/Python/blob/main/SR_GammaFlip_Interativo_BARCHART.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### BIBLIOTECA

In [69]:
### CÓDIGO ADAPTADO PARA BARCHART - ESTRUTURA ORIGINAL COMPLETA ###

# ========== CÉLULA 1: INSTALAR PLOTLY ==========
### Rodar essa célula somente uma vez ###
#!pip install plotly

# ========================================
# CÉLULA 3 - IMPORTS
# ========================================
import pandas as pd
import plotly
pd.set_option('plotting.backend','plotly')
import plotly.graph_objs as go
import numpy as np
import scipy
from scipy.stats import norm
#import matplotlib.pyplot as plt
import calendar
from datetime import datetime, timedelta, date

### Arquivo CSV

In [70]:
# ========================================
# CÉLULA 4 - FORMATO DISPLAY
# ========================================
pd.options.display.float_format = '{:,.4f}'.format

In [71]:
# ========================================
# CÉLULA 6 - FUNÇÕES E PARÂMETROS
# ========================================
# Parametros de entrada
filename = 'nqz25-volatility-greeks-exp-12_19_25-50-strikes-_-10-29-2025.csv'

In [72]:
# Black-Scholes European-Options Gamma
def calcGammaEx(S, K, vol, T, r, q, optType, OI):
    if T == 0 or vol == 0:
        return 0

    dp = (np.log(S/K) + (r - q + 0.5*vol**2)*T) / (vol*np.sqrt(T))
    dm = dp - vol*np.sqrt(T)

    if optType == 'call':
        gamma = np.exp(-q*T) * norm.pdf(dp) / (S * vol * np.sqrt(T))
        return OI * 100 * S * S * 0.01 * gamma
    else:
        gamma = K * np.exp(-r*T) * norm.pdf(dm) / (S * S * vol * np.sqrt(T))
        return OI * 100 * S * S * 0.01 * gamma

def isThirdFriday(d):
    return d.weekday() == 4 and 15 <= d.day <= 21

In [97]:
# ========================================
# CÉLULA 7 - MARKDOWN
# ========================================
# ### PREPARAÇÃO DO ARQUIVO

# ========================================
# CÉLULA 8 - PROCESSAR CSV BARCHART
# ========================================
# ADAPTAÇÃO: Ler CSV do Barchart
df_raw = pd.read_csv(filename)
df_raw = df_raw[df_raw['Type'].notna()].copy()

# DEFINA O SPOT PRICE MANUALMENTE (Barchart não inclui no CSV)
spotPrice = 26200.00  # ⚠️ AJUSTE AQUI O PREÇO ATUAL DO NQ

fromStrike = 0.8 * spotPrice
toStrike = 1.2 * spotPrice

# Data de hoje
todayDate = datetime.now()

# Limpar colunas do Barchart
df_raw['Strike'] = df_raw['Strike'].str.replace(',', '').astype(float)
df_raw['IV'] = df_raw['IV'].str.replace('%', '').astype(float) / 100

# ⚠️ IMPORTANTE: Adicionar Open Interest (Barchart não tem)
df_raw['OpenInt'] = 100  # AJUSTE AQUI se tiver OI de outra fonte

# Separar calls e puts
df_calls = df_raw[df_raw['Type'] == 'Call'].copy()
df_puts = df_raw[df_raw['Type'] == 'Put'].copy()

# Data de expiração (ajuste conforme seu CSV)
expiration_date = datetime(2025, 12, 19, 16, 0)  # ⚠️ AJUSTE A DATA DE VENCIMENTO

# Criar DataFrame no formato original (linha por strike com call e put)
strikes_unique = sorted(df_raw['Strike'].unique())
data_list = []

for strike in strikes_unique:
    call_row = df_calls[df_calls['Strike'] == strike]
    put_row = df_puts[df_puts['Strike'] == strike]

    row_data = {
        'ExpirationDate': expiration_date,
        'StrikePrice': strike,
        'CallIV': call_row['IV'].values[0] if len(call_row) > 0 else 0,
        'PutIV': put_row['IV'].values[0] if len(put_row) > 0 else 0,
        'CallGamma': call_row['Gamma'].values[0] if len(call_row) > 0 else 0,
        'PutGamma': put_row['Gamma'].values[0] if len(put_row) > 0 else 0,
        'CallOpenInt': call_row['OpenInt'].values[0] if len(call_row) > 0 else 0,
        'PutOpenInt': put_row['OpenInt'].values[0] if len(put_row) > 0 else 0,
        'CallDelta': call_row['Delta'].values[0] if len(call_row) > 0 else 0,
        'PutDelta': put_row['Delta'].values[0] if len(put_row) > 0 else 0,
    }
    data_list.append(row_data)

df = pd.DataFrame(data_list)

# Calcular dias até expiração (em dias úteis / 262)
df['daysTillExp'] = df['ExpirationDate'].apply(
    lambda x: np.busday_count(todayDate.date(), x.date()) / 262
)

FileNotFoundError: [Errno 2] No such file or directory: 'nqz25-volatility-greeks-exp-12_19_25-50-strikes-_-10-29-2025.csv'

### GAMMA - GEX

In [76]:
# ========================================
# CÉLULA 9 - MARKDOWN
# ========================================
# ### GAMMA - GEX

# ========================================
# CÉLULA 10 - CALCULAR GEX
# ========================================
# ---=== CALCULATE SPOT GAMMA ===---
df['CallGEX'] = df['CallGamma'] * df['CallOpenInt'] * 100 * spotPrice * spotPrice * 0.01
df['PutGEX'] = df['PutGamma'] * df['PutOpenInt'] * 100 * spotPrice * spotPrice * 0.01 * -1

df['TotalGamma'] = (df.CallGEX + df.PutGEX) / 10**9
dfAgg = df.groupby(['StrikePrice']).sum(numeric_only=True)
strikes = dfAgg.index.values

In [78]:
# ========================================
# CÉLULA 11 - GRÁFICO 1: ABSOLUTE GAMMA EXPOSURE
# ========================================
# Chart 1: Absolute Gamma Exposure
x_data = strikes
y_data = dfAgg['TotalGamma'].to_numpy()

fig = go.Figure(
    go.Bar(
        x=x_data,
        y=y_data,
        width=6,
        marker_color='rgb(26, 118, 255)',
        marker_line_color='black',
        marker_line_width=0.15,
        name='Gamma Exposure'
    )
)

fig.add_shape(
    type='line',
    x0=spotPrice,
    y0=min(y_data),
    x1=spotPrice,
    y1=max(y_data),
    line=dict(color='red', width=2, dash='dash')
)

fig.update_layout(
    title={
        'text': f"Total Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% Ativo Move",
        'font': {'size': 20, 'family': 'Arial Black',}
    },
    xaxis_title='Strike',
    yaxis_title='Spot Gamma Exposure ($ billions/1% move)',
    xaxis=dict(range=[fromStrike, toStrike]),
    yaxis=dict(tickformat='$,.2f'),
    plot_bgcolor='white',
    font=dict(family='Arial', size=12, color='black')
)

fig.update_layout(width=1750, height=800)
fig.show()

In [79]:
# ========================================
# CÉLULA 12 - ANÁLISE GRÁFICO 1
# ========================================
# DADOS DO CHART 1
dfAgg_sorted = dfAgg.sort_values(by='TotalGamma')
# Get the 3 smallest gamma values
smallest_gamma = dfAgg_sorted.head(3)
print("Menores valores de Gamma:")
print(f"  Put Wall: {smallest_gamma.iloc[0]['TotalGamma']:.4f} at strike {smallest_gamma.index[0]:.2f}")
print(f"  Large Gamma: {smallest_gamma.iloc[1]['TotalGamma']:.4f} at strike {smallest_gamma.index[1]:.2f}")
print(f"  Large Gamma: {smallest_gamma.iloc[2]['TotalGamma']:.4f} at strike {smallest_gamma.index[2]:.2f}")

# Get the 3 largest gamma values
largest_gamma = dfAgg_sorted.tail(3)
print("\nMaiores valores de Gamma:")
print(f"  Call Wall: {largest_gamma.iloc[2]['TotalGamma']:.4f} at strike {largest_gamma.index[2]:.2f}")
print(f"  Large Gamma: {largest_gamma.iloc[1]['TotalGamma']:.4f} at strike {largest_gamma.index[1]:.2f}")
print(f"  Large Gamma: {largest_gamma.iloc[0]['TotalGamma']:.4f} at strike {largest_gamma.index[0]:.2f}")


Menores valores de Gamma:
  Put Wall: -0.0038 at strike 23300.00
  Large Gamma: -0.0037 at strike 23200.00
  Large Gamma: -0.0037 at strike 23250.00

Maiores valores de Gamma:
  Call Wall: 0.0056 at strike 25900.00
  Large Gamma: 0.0036 at strike 26400.00
  Large Gamma: 0.0036 at strike 26100.00


In [80]:
# ========================================
# CÉLULA 13 - GRÁFICO 2: GAMMA BY CALLS AND PUTS
# ========================================
# Chart 2: Absolute Gamma Exposure by Calls and Puts
fig = go.Figure()
fig.add_bar(x=strikes, y=dfAgg['CallGEX'].to_numpy() / 10**9, width=6, name="Call Gamma")
fig.add_bar(x=strikes, y=dfAgg['PutGEX'].to_numpy() / 10**9, width=6, name="Put Gamma")
fig.update_xaxes(range=[fromStrike, toStrike])

chartTitle = "Total Gamma: $" + str("{:.2f}".format(df['TotalGamma'].sum())) + " Bn per 1% SPX Move"
fig.update_layout(title_text=chartTitle, title_font=dict(size=20, family="Arial Black"))
fig.update_xaxes(title_text="Strike")
fig.update_yaxes(title_text="Spot Gamma Exposure ($ billions/1% move)")

fig.add_shape(dict(
    type="line",
    x0=spotPrice,
    y0=0,
    x1=spotPrice,
    y1=max(dfAgg['CallGEX'].to_numpy() / 10**9),
    line=dict(color="black", width=2),
    name="SPX Spot:" + str("{:,.0f}".format(spotPrice))
))

fig.update_layout(width=1750, height=800)
fig.show()


In [83]:
# ========================================
# CÉLULA 14 - ANÁLISE GRÁFICO 2
# ========================================
# DADOS CHART 2
dfAgg['AbsoluteTotalGEX'] = dfAgg['CallGEX'].abs() + dfAgg['PutGEX'].abs()
# Sort by AbsoluteTotalGEX to find the strikes with the largest combined exposure
dfAgg_sorted_gex = dfAgg.sort_values(by='AbsoluteTotalGEX', ascending=False)
# Get the top 6 strikes based on combined absolute GEX
gex_levels = dfAgg_sorted_gex.head(6)
print("Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):")
for idx, strike in enumerate(gex_levels.index, 1):
    print(f"  {idx}. Strike {strike:.2f}: Call GEX = {gex_levels.loc[strike, 'CallGEX']/10**9:.4f}, Put GEX = {gex_levels.loc[strike, 'PutGEX']/10**9:.4f}, Total Absolute = {gex_levels.loc[strike, 'AbsoluteTotalGEX']/10**9:.4f}")

Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):
  1. Strike 26300.00: Call GEX = 0.0171, Put GEX = -0.0141, Total Absolute = 0.0312
  2. Strike 25900.00: Call GEX = 0.0182, Put GEX = -0.0126, Total Absolute = 0.0307
  3. Strike 26200.00: Call GEX = 0.0166, Put GEX = -0.0135, Total Absolute = 0.0301
  4. Strike 26400.00: Call GEX = 0.0168, Put GEX = -0.0132, Total Absolute = 0.0300
  5. Strike 26100.00: Call GEX = 0.0168, Put GEX = -0.0132, Total Absolute = 0.0300
  6. Strike 26000.00: Call GEX = 0.0154, Put GEX = -0.0136, Total Absolute = 0.0290


In [84]:
# ========================================
# CÉLULA 15 - CALCULAR GAMMA PROFILE
# ========================================
# For each spot level, calculate gamma exposure by applying gamma exposure at that strike
levels = np.linspace(fromStrike, toStrike, 60)

# Expiração próxima e mensal
nextExpiry = df['ExpirationDate'].min()
df["isThirdFriday"] = df['ExpirationDate'].apply(isThirdFriday)
thirdFridays = df.loc[df["isThirdFriday"] == True]
nextMonthlyExp = thirdFridays['ExpirationDate'].min() if len(thirdFridays) > 0 else nextExpiry

totalGamma = []
totalGammaExNext = []
totalGammaExFri = []

# Para cada nível de preço
for level in levels:
    df['callGammaEx'] = df.apply(lambda row: calcGammaEx(
        level, row['StrikePrice'], row['CallIV'],
        (row['ExpirationDate'] - todayDate).days / 365,
        0, 0, 'call', row['CallOpenInt']
    ), axis=1)

    df['putGammaEx'] = df.apply(lambda row: calcGammaEx(
        level, row['StrikePrice'], row['PutIV'],
        (row['ExpirationDate'] - todayDate).days / 365,
        0, 0, 'put', row['PutOpenInt']
    ), axis=1)

    totalGamma.append((df['callGammaEx'].sum() - df['putGammaEx'].sum()) / 10**9)

    # Ex-Next Expiry
    totalGammaExNext.append(
        (df.loc[df['ExpirationDate'] != nextExpiry, 'callGammaEx'].sum() -
         df.loc[df['ExpirationDate'] != nextExpiry, 'putGammaEx'].sum()) / 10**9
    )

    # Ex-Next Monthly
    totalGammaExFri.append(
        (df.loc[df['ExpirationDate'] != nextMonthlyExp, 'callGammaEx'].sum() -
         df.loc[df['ExpirationDate'] != nextMonthlyExp, 'putGammaEx'].sum()) / 10**9
    )


In [85]:
# ========================================
# CÉLULA 16 - GRÁFICO 3: GAMMA EXPOSURE PROFILE
# ========================================
# Chart 3: Gamma Exposure Profile
fig = go.Figure()

fig.add_trace(go.Scatter(x=levels, y=totalGamma, mode='lines', name='All Expiries'))
fig.add_trace(go.Scatter(x=levels, y=totalGammaExNext, mode='lines', name='Ex-Next Expiry'))
fig.add_trace(go.Scatter(x=levels, y=totalGammaExFri, mode='lines', name='Ex-Next Monthly Expiry'))

chartTitle = "Gamma Exposure Profile, SPX, " + todayDate.strftime('%d %b %Y')
fig.update_layout(
    title=chartTitle,
    xaxis_title='Index Price',
    yaxis_title='Gamma Exposure ($ billions/1% move)',
    title_font=dict(size=20, family="Arial Black")
)

fig.add_shape(dict(
    type="line",
    x0=spotPrice,
    y0=min(totalGamma),
    x1=spotPrice,
    y1=max(totalGamma),
    line=dict(color="red", width=2, dash="dash")
))

fig.update_layout(width=1750, height=800, plot_bgcolor='white')
fig.show()


In [92]:
# ========================================
# CÉLULA 17 - ANÁLISE GRÁFICO 3
# ========================================
# DADOS CHART 3
# Find the point on the 'Ex-Next Monthly Expiry' line closest to the spot price (green line and red dotted line intersection as defined by user)
closest_level_index_green_red = np.abs(levels - spotPrice).argmin()
gamma_at_spot_green_red = totalGammaExFri[closest_level_index_green_red]
gamma_at_spot_strike_green_red = levels[closest_level_index_green_red]

print(f"Gamma Flip (Ex-Next Monthly Expiry): {gamma_at_spot_green_red:.4f} at strike {gamma_at_spot_strike_green_red:.2f} (Intersection of Green and Red lines)")

Gamma Flip (Ex-Next Monthly Expiry): 0.0000 at strike 26111.19 (Intersection of Green and Red lines)


In [94]:
# ========================================
# CÉLULA 18 - CONSOLIDAÇÃO RESULTADOS GAMMA
# ========================================
# Consolidating results from CHART 1, CHART 2, and CHART 3
print("--- DADOS CHART 1 ---")
# DADOS CHART 1
dfAgg_sorted = dfAgg.sort_values(by='TotalGamma')
smallest_gamma = dfAgg_sorted.head(3)
print("Menores valores de Gamma:")
print(f"  Put Wall: {smallest_gamma.iloc[0]['TotalGamma']:.4f} at strike {smallest_gamma.index[0]:.2f}")
print(f"  Large Gamma: {smallest_gamma.iloc[1]['TotalGamma']:.4f} at strike {smallest_gamma.index[1]:.2f}")
print(f"  Large Gamma: {smallest_gamma.iloc[2]['TotalGamma']:.4f} at strike {smallest_gamma.index[2]:.2f}")

largest_gamma = dfAgg_sorted.tail(3)
print("\nMaiores valores de Gamma:")
print(f"  Call Wall: {largest_gamma.iloc[2]['TotalGamma']:.4f} at strike {largest_gamma.index[2]:.2f}")
print(f"  Large Gamma: {largest_gamma.iloc[1]['TotalGamma']:.4f} at strike {largest_gamma.index[1]:.2f}")
print(f"  Large Gamma: {largest_gamma.iloc[0]['TotalGamma']:.4f} at strike {largest_gamma.index[0]:.2f}")

print("\n--- DADOS CHART 2 ---")
print("Top 6 GEX Levels:")
for idx, strike in enumerate(gex_levels.index, 1):
    print(f"  {idx}. Strike {strike:.2f}: Total Absolute = {gex_levels.loc[strike, 'AbsoluteTotalGEX']/10**9:.4f}")

print("\n--- DADOS CHART 3 ---")
# Include Gamma Flip from CÉLULA 17
print(f"Gamma Flip (Ex-Next Monthly Expiry): {gamma_at_spot_green_red:.4f} at strike {gamma_at_spot_strike_green_red:.2f}")

--- DADOS CHART 1 ---
Menores valores de Gamma:
  Put Wall: -0.0038 at strike 23300.00
  Large Gamma: -0.0037 at strike 23200.00
  Large Gamma: -0.0037 at strike 23250.00

Maiores valores de Gamma:
  Call Wall: 0.0056 at strike 25900.00
  Large Gamma: 0.0036 at strike 26400.00
  Large Gamma: 0.0036 at strike 26100.00

--- DADOS CHART 2 ---
Top 6 GEX Levels:
  1. Strike 26300.00: Total Absolute = 0.0312
  2. Strike 25900.00: Total Absolute = 0.0307
  3. Strike 26200.00: Total Absolute = 0.0301
  4. Strike 26400.00: Total Absolute = 0.0300
  5. Strike 26100.00: Total Absolute = 0.0300
  6. Strike 26000.00: Total Absolute = 0.0290

--- DADOS CHART 3 ---
Gamma Flip (Ex-Next Monthly Expiry): 0.0000 at strike 26111.19


In [96]:
# ========================================
# CÉLULA 19 - FILTRO 5 DTE (GAMMA)
# ========================================
# --- Consolidating results for 5 DTE only ---
print("\n--- DADOS FILTRADOS PARA 5 DTE ---")
# Filter data for 5 DTE
df_5dte = df[df['daysTillExp'] <= 5/262].copy()

# Recalculate aggregated data for 5 DTE
dfAgg_5dte = df_5dte.groupby(['StrikePrice']).sum(numeric_only=True)

print("\n--- DADOS CHART 1 (5 DTE) ---")
if len(dfAgg_5dte) > 0:
    dfAgg_sorted_5dte = dfAgg_5dte.sort_values(by='TotalGamma')
    smallest_gamma_5dte = dfAgg_sorted_5dte.head(3)
    print("Menores valores de Gamma (5 DTE):")
    for i in range(min(3, len(smallest_gamma_5dte))):
        print(f"  {smallest_gamma_5dte.iloc[i]['TotalGamma']:.4f} at strike {smallest_gamma_5dte.index[i]:.2f}")

    largest_gamma_5dte = dfAgg_sorted_5dte.tail(3)
    print("\nMaiores valores de Gamma (5 DTE):")
    for i in range(min(3, len(largest_gamma_5dte))-1, -1, -1):
        print(f"  {largest_gamma_5dte.iloc[i]['TotalGamma']:.4f} at strike {largest_gamma_5dte.index[i]:.2f}")
else:
    print("Nenhum dado para 5 DTE")


--- DADOS FILTRADOS PARA 5 DTE ---


KeyError: 'daysTillExp'

In [98]:
# ========================================
# CÉLULA 20 - FILTRO 0 DTE (GAMMA)
# ========================================
# --- Consolidating results for 0 DTE only ---
print("\n--- DADOS FILTRADOS PARA 0 DTE ---")
# Filter data for 0 DTE
df_0dte = df[df['daysTillExp'] <= 1/262].copy()

# Recalculate aggregated data for 0 DTE
dfAgg_0dte = df_0dte.groupby(['StrikePrice']).sum(numeric_only=True)

print("\n--- DADOS CHART 1 (0 DTE) ---")
if len(dfAgg_0dte) > 0:
    dfAgg_sorted_0dte = dfAgg_0dte.sort_values(by='TotalGamma')
    smallest_gamma_0dte = dfAgg_sorted_0dte.head(3)
    print("Menores valores de Gamma (0 DTE):")
    for i in range(min(3, len(smallest_gamma_0dte))):
        print(f"  {smallest_gamma_0dte.iloc[i]['TotalGamma']:.4f} at strike {smallest_gamma_0dte.index[i]:.2f}")

    largest_gamma_0dte = dfAgg_sorted_0dte.tail(3)
    print("\nMaiores valores de Gamma (0 DTE):")
    for i in range(min(3, len(largest_gamma_0dte))-1, -1, -1):
        print(f"  {largest_gamma_0dte.iloc[i]['TotalGamma']:.4f} at strike {largest_gamma_0dte.index[i]:.2f}")
else:
    print("Nenhum dado para 0 DTE")


--- DADOS FILTRADOS PARA 0 DTE ---


KeyError: 'daysTillExp'

### DELTA - DEX

In [99]:
# ========================================
# CÉLULA 21 - MARKDOWN
# ========================================
# ### DELTA - DEX

# ========================================
# CÉLULA 22 - CALCULAR DEX
# ========================================
# ---=== CALCULATE SPOT DELTA ===---
df['CallDEX'] = df['CallDelta'] * df['CallOpenInt'] * 100 * spotPrice * 0.01
df['PutDEX'] = df['PutDelta'] * df['PutOpenInt'] * 100 * spotPrice * 0.01

df['TotalDelta'] = (df.CallDEX + df.PutDEX) / 10**6
dfAgg_delta = df.groupby(['StrikePrice']).sum(numeric_only=True)
strikes_delta = dfAgg_delta.index.values

In [100]:
# ========================================
# CÉLULA 23 - GRÁFICO 4: ABSOLUTE DELTA EXPOSURE
# ========================================
# Chart 4: Absolute Delta Exposure
x_data_delta = strikes_delta
y_data_delta = dfAgg_delta['TotalDelta'].to_numpy()

fig_delta4 = go.Figure(
    go.Bar(
        x=x_data_delta,
        y=y_data_delta,
        width=6,
        marker_color='rgb(26, 118, 255)',
        marker_line_color='black',
        marker_line_width=0.15,
        name='Delta Exposure'
    )
)

fig_delta4.add_shape(
    type='line',
    x0=spotPrice,
    y0=min(y_data_delta),
    x1=spotPrice,
    y1=max(y_data_delta),
    line=dict(color='red', width=2, dash='dash')
)

fig_delta4.update_layout(
    title={
        'text': f"Total Delta: ${df['TotalDelta'].sum():,.2f} Million per 1% Ativo Move",
        'font': {'size': 20, 'family': 'Arial Black'}
    },
    xaxis_title='Strike',
    yaxis_title='Spot Delta Exposure ($ millions/1% move)',
    xaxis=dict(range=[fromStrike, toStrike]),
    yaxis=dict(tickformat='$,.2f'),
    plot_bgcolor='white',
    font=dict(family='Arial', size=12, color='black')
)

fig_delta4.update_layout(width=1750, height=800)
fig_delta4.show()

In [101]:
# ========================================
# CÉLULA 24 - GRÁFICO 5: DELTA BY CALLS AND PUTS
# ========================================
# Chart 5: Absolute Delta Exposure by Calls and Puts
fig_delta5 = go.Figure()
fig_delta5.add_bar(x=strikes_delta, y=dfAgg_delta['CallDEX'].to_numpy() / 10**6, width=6, name="Call Delta")
fig_delta5.add_bar(x=strikes_delta, y=dfAgg_delta['PutDEX'].to_numpy() / 10**6, width=6, name="Put Delta")
fig_delta5.update_xaxes(range=[fromStrike, toStrike])

chartTitle_delta5 = "Total Delta: $" + str("{:.2f}".format(df['TotalDelta'].sum())) + " Million per 1% SPX Move"
fig_delta5.update_layout(title_text=chartTitle_delta5, title_font=dict(size=20, family="Arial Black"))
fig_delta5.update_xaxes(title_text="Strike")
fig_delta5.update_yaxes(title_text="Spot Delta Exposure ($ millions/1% move)")

fig_delta5.add_shape(dict(
    type="line",
    x0=spotPrice,
    y0=min(dfAgg_delta['PutDEX'].to_numpy() / 10**6),
    x1=spotPrice,
    y1=max(dfAgg_delta['CallDEX'].to_numpy() / 10**6),
    line=dict(color="black", width=2)
))

fig_delta5.update_layout(width=1750, height=800)
fig_delta5.show()

In [102]:
# ========================================
# CÉLULA 25 - CALCULAR DELTA PROFILE
# ========================================
levels_delta = np.linspace(fromStrike, toStrike, 60)

totalDelta = []
totalDeltaExNext = []
totalDeltaExFri = []

for level in levels_delta:
    # Para Delta, usamos o delta já calculado (simplificação)
    totalDelta.append(df['TotalDelta'].sum())
    totalDeltaExNext.append(
        df.loc[df['ExpirationDate'] != nextExpiry, 'TotalDelta'].sum()
    )
    totalDeltaExFri.append(
        df.loc[df['ExpirationDate'] != nextMonthlyExp, 'TotalDelta'].sum()
    )


In [103]:
# ========================================
# CÉLULA 26 - GRÁFICO 6: DELTA EXPOSURE PROFILE
# ========================================
# Chart 6: Delta Exposure Profile
fig_delta6 = go.Figure()

fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDelta, mode='lines', name='All Expiries'))
fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDeltaExNext, mode='lines', name='Ex-Next Expiry'))
fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDeltaExFri, mode='lines', name='Ex-Next Monthly Expiry'))

chartTitle_delta6 = "Delta Exposure Profile, SPX, " + todayDate.strftime('%d %b %Y')
fig_delta6.update_layout(
    title=chartTitle_delta6,
    xaxis_title='Index Price',
    yaxis_title='Delta Exposure ($ millions/1% move)',
    title_font=dict(size=20, family="Arial Black")
)

fig_delta6.add_shape(dict(
    type="line",
    x0=spotPrice,
    y0=min(totalDelta),
    x1=spotPrice,
    y1=max(totalDelta),
    line=dict(color="red", width=2, dash="dash")
))

fig_delta6.update_layout(width=1750, height=800, plot_bgcolor='white')
fig_delta6.show()


In [104]:
# ========================================
# CÉLULA 27 - ANÁLISE DELTA
# ========================================
# Consolidating results from CHART 4, CHART 5, and CHART 6
print("--- DADOS CHART 4 ---")
# Find the 3 strikes with the smallest (most negative) total Delta Exposure
smallest_delta = dfAgg_delta.sort_values(by='TotalDelta').head(3)
print("Menores valores de Delta:")
print(f"  Put Delta Wall: {smallest_delta.iloc[0]['TotalDelta']:.4f} at strike {smallest_delta.index[0]:.2f}")
print(f"  Large Delta: {smallest_delta.iloc[1]['TotalDelta']:.4f} at strike {smallest_delta.index[1]:.2f}")
print(f"  Large Delta: {smallest_delta.iloc[2]['TotalDelta']:.4f} at strike {smallest_delta.index[2]:.2f}")

# Find the 3 strikes with the largest (most positive) total Delta Exposure
largest_delta = dfAgg_delta.sort_values(by='TotalDelta').tail(3)
print("\nMaiores valores de Delta:")
print(f"  Call Delta Wall: {largest_delta.iloc[2]['TotalDelta']:.4f} at strike {largest_delta.index[2]:.2f}")
print(f"  Large Delta: {largest_delta.iloc[1]['TotalDelta']:.4f} at strike {largest_delta.index[1]:.2f}")
print(f"  Large Delta: {largest_delta.iloc[0]['TotalDelta']:.4f} at strike {largest_delta.index[0]:.2f}")


--- DADOS CHART 4 ---
Menores valores de Delta:
  Put Delta Wall: -2.3695 at strike 35500.00
  Large Delta: -2.3644 at strike 35000.00
  Large Delta: -2.3586 at strike 34500.00

Maiores valores de Delta:
  Call Delta Wall: 2.4483 at strike 22250.00
  Large Delta: 2.4436 at strike 22200.00
  Large Delta: 2.4354 at strike 22300.00


In [105]:
# ========================================
# CÉLULA 28 - FILTRO 5 DTE (DELTA)
# ========================================
# --- Consolidating results for 5 DTE only (Delta) ---
print("\n--- DADOS FILTRADOS PARA 5 DTE (Delta) ---")
df_5dte_delta = df[df['daysTillExp'] <= 5/262].copy()
dfAgg_5dte_delta = df_5dte_delta.groupby(['StrikePrice']).sum(numeric_only=True)

if len(dfAgg_5dte_delta) > 0:
    smallest_delta_5dte = dfAgg_5dte_delta.sort_values(by='TotalDelta').head(3)
    print("Menores valores de Delta (5 DTE):")
    for i in range(min(3, len(smallest_delta_5dte))):
        print(f"  {smallest_delta_5dte.iloc[i]['TotalDelta']:.4f} at strike {smallest_delta_5dte.index[i]:.2f}")

    largest_delta_5dte = dfAgg_5dte_delta.sort_values(by='TotalDelta').tail(3)
    print("\nMaiores valores de Delta (5 DTE):")
    for i in range(min(3, len(largest_delta_5dte))-1, -1, -1):
        print(f"  {largest_delta_5dte.iloc[i]['TotalDelta']:.4f} at strike {largest_delta_5dte.index[i]:.2f}")
else:
    print("Nenhum dado para 5 DTE")



--- DADOS FILTRADOS PARA 5 DTE (Delta) ---


KeyError: 'daysTillExp'

In [106]:
# ========================================
# CÉLULA 29 - FILTRO 0 DTE (DELTA)
# ========================================
# --- Consolidating results for 0 DTE only (Delta) ---
print("\n--- DADOS FILTRADOS PARA 0 DTE (Delta) ---")
df_0dte_delta = df[df['daysTillExp'] <= 1/262].copy()
dfAgg_0dte_delta = df_0dte_delta.groupby(['StrikePrice']).sum(numeric_only=True)

if len(dfAgg_0dte_delta) > 0:
    smallest_delta_0dte = dfAgg_0dte_delta.sort_values(by='TotalDelta').head(3)
    print("Menores valores de Delta (0 DTE):")
    for i in range(min(3, len(smallest_delta_0dte))):
        print(f"  {smallest_delta_0dte.iloc[i]['TotalDelta']:.4f} at strike {smallest_delta_0dte.index[i]:.2f}")

    largest_delta_0dte = dfAgg_0dte_delta.sort_values(by='TotalDelta').tail(3)
    print("\nMaiores valores de Delta (0 DTE):")
    for i in range(min(3, len(largest_delta_0dte))-1, -1, -1):
        print(f"  {largest_delta_0dte.iloc[i]['TotalDelta']:.4f} at strike {largest_delta_0dte.index[i]:.2f}")
else:
    print("Nenhum dado para 0 DTE")


--- DADOS FILTRADOS PARA 0 DTE (Delta) ---


KeyError: 'daysTillExp'

### RESULTADOS GAMMA

In [107]:
# ========================================
# CÉLULA 31 - GERADOR TRADINGVIEW - GAMMA (ALL EXPIRIES)
# ========================================
# ==================== CÉLULA FINAL DO NOTEBOOK ====================
# Cole esta célula no final do seu notebook Jupyter/Colab
# Ela irá gerar UMA ÚNICA LINHA para atualizar tudo de uma vez

print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP (VERSÃO SIMPLIFICADA)")
print("="*80 + "\n")

# ==================== COLETA DOS DADOS ====================
# CHART 1 - Spot Gamma Levels
dfAgg_sorted = dfAgg.sort_values(by='TotalGamma')
smallest_gamma = dfAgg_sorted.head(3)
largest_gamma = dfAgg_sorted.tail(3)

put_wall = smallest_gamma.index[0]
call_wall = largest_gamma.index[2]

# CHART 2 - GEX Levels
dfAgg['AbsoluteTotalGEX'] = dfAgg['CallGEX'].abs() + dfAgg['PutGEX'].abs()
dfAgg_sorted_gex = dfAgg.sort_values(by='AbsoluteTotalGEX', ascending=False)
gex_levels = dfAgg_sorted_gex.head(6)

gex_level_1 = gex_levels.index[0] if len(gex_levels) > 0 else 0
gex_level_2 = gex_levels.index[1] if len(gex_levels) > 1 else 0
gex_level_3 = gex_levels.index[2] if len(gex_levels) > 2 else 0
gex_level_4 = gex_levels.index[3] if len(gex_levels) > 3 else 0
gex_level_5 = gex_levels.index[4] if len(gex_levels) > 4 else 0
gex_level_6 = gex_levels.index[5] if len(gex_levels) > 5 else 0

# CHART 3 - Gamma Flip
gamma_flip_index = np.where(np.array(totalGammaExFri) > 0)[0][0] if np.any(np.array(totalGammaExFri) > 0) else None
gamma_flip = levels[gamma_flip_index] if gamma_flip_index is not None else 0

# ==================== GERAR CÓDIGO TRADINGVIEW ====================
tradingview_code = f'''
// GEX Levels
gex_level_1 = {gex_level_1:.2f}
gex_level_2 = {gex_level_2:.2f}
gex_level_3 = {gex_level_3:.2f}
gex_level_4 = {gex_level_4:.2f}
gex_level_5 = {gex_level_5:.2f}
gex_level_6 = {gex_level_6:.2f}

// Gamma Walls
put_wall = {put_wall:.2f}
call_wall = {call_wall:.2f}

// Gamma Flip
gamma_flip = {gamma_flip:.2f}

// Plot lines
plot(gex_level_1, "GEX Level 1", color=color.yellow, linewidth=2)
plot(gex_level_2, "GEX Level 2", color=color.yellow, linewidth=2)
plot(gex_level_3, "GEX Level 3", color=color.yellow, linewidth=2)
plot(gex_level_4, "GEX Level 4", color=color.yellow, linewidth=2)
plot(gex_level_5, "GEX Level 5", color=color.yellow, linewidth=2)
plot(gex_level_6, "GEX Level 6", color=color.yellow, linewidth=2)
plot(put_wall, "Put Wall", color=color.red, linewidth=3)
plot(call_wall, "Call Wall", color=color.green, linewidth=3)
plot(gamma_flip, "Gamma Flip", color=color.orange, linewidth=3)
'''

print("📋 CÓDIGO PINE SCRIPT PARA TRADINGVIEW:")
print(tradingview_code)

print("\n✅ Copie o código acima e cole no TradingView Pine Editor!")



🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP (VERSÃO SIMPLIFICADA)

📋 CÓDIGO PINE SCRIPT PARA TRADINGVIEW:

// GEX Levels
gex_level_1 = 26300.00
gex_level_2 = 25900.00
gex_level_3 = 26200.00
gex_level_4 = 26400.00
gex_level_5 = 26100.00
gex_level_6 = 26000.00

// Gamma Walls
put_wall = 23300.00
call_wall = 25900.00

// Gamma Flip
gamma_flip = 0.00

// Plot lines
plot(gex_level_1, "GEX Level 1", color=color.yellow, linewidth=2)
plot(gex_level_2, "GEX Level 2", color=color.yellow, linewidth=2)
plot(gex_level_3, "GEX Level 3", color=color.yellow, linewidth=2)
plot(gex_level_4, "GEX Level 4", color=color.yellow, linewidth=2)
plot(gex_level_5, "GEX Level 5", color=color.yellow, linewidth=2)
plot(gex_level_6, "GEX Level 6", color=color.yellow, linewidth=2)
plot(put_wall, "Put Wall", color=color.red, linewidth=3)
plot(call_wall, "Call Wall", color=color.green, linewidth=3)
plot(gamma_flip, "Gamma Flip", color=color.orange, linewidth=3)


✅ Copie o código acima e cole no TradingView Pine Edit

In [108]:
# ========================================
# CÉLULA 32 - GERADOR TRADINGVIEW - GAMMA (5 DTE)
# ========================================
# ==================== CÉLULA FINAL DO NOTEBOOK (5 DTE) ====================
print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP (5 DTE - VERSÃO SIMPLIFICADA)")
print("="*80 + "\n")

# ==================== COLETA DOS DADOS (5 DTE) ====================
if len(dfAgg_5dte) > 0:
    # CHART 1 - Spot Gamma Levels (5 DTE)
    dfAgg_sorted_5dte = dfAgg_5dte.sort_values(by='TotalGamma')
    smallest_gamma_5dte = dfAgg_sorted_5dte.head(3)
    largest_gamma_5dte = dfAgg_sorted_5dte.tail(3)

    put_wall_5dte = smallest_gamma_5dte.index[0] if len(smallest_gamma_5dte) > 0 else 0
    call_wall_5dte = largest_gamma_5dte.index[-1] if len(largest_gamma_5dte) > 0 else 0

    # CHART 2 - GEX Levels (5 DTE)
    dfAgg_5dte['AbsoluteTotalGEX'] = dfAgg_5dte['CallGEX'].abs() + dfAgg_5dte['PutGEX'].abs()
    dfAgg_sorted_gex_5dte = dfAgg_5dte.sort_values(by='AbsoluteTotalGEX', ascending=False)
    gex_levels_5dte = dfAgg_sorted_gex_5dte.head(6)

    gex_level_1_5dte = gex_levels_5dte.index[0] if len(gex_levels_5dte) > 0 else 0
    gex_level_2_5dte = gex_levels_5dte.index[1] if len(gex_levels_5dte) > 1 else 0
    gex_level_3_5dte = gex_levels_5dte.index[2] if len(gex_levels_5dte) > 2 else 0
    gex_level_4_5dte = gex_levels_5dte.index[3] if len(gex_levels_5dte) > 3 else 0
    gex_level_5_5dte = gex_levels_5dte.index[4] if len(gex_levels_5dte) > 4 else 0
    gex_level_6_5dte = gex_levels_5dte.index[5] if len(gex_levels_5dte) > 5 else 0

    # ==================== GERAR CÓDIGO TRADINGVIEW (5 DTE) ====================
    tradingview_code_5dte = f'''
// GEX Levels (5 DTE)
gex_level_1_5dte = {gex_level_1_5dte:.2f}
gex_level_2_5dte = {gex_level_2_5dte:.2f}
gex_level_3_5dte = {gex_level_3_5dte:.2f}
gex_level_4_5dte = {gex_level_4_5dte:.2f}
gex_level_5_5dte = {gex_level_5_5dte:.2f}
gex_level_6_5dte = {gex_level_6_5dte:.2f}

// Gamma Walls (5 DTE)
put_wall_5dte = {put_wall_5dte:.2f}
call_wall_5dte = {call_wall_5dte:.2f}

// Plot lines
plot(gex_level_1_5dte, "GEX Level 1 (5D)", color=color.new(color.yellow, 30), linewidth=2)
plot(gex_level_2_5dte, "GEX Level 2 (5D)", color=color.new(color.yellow, 30), linewidth=2)
plot(gex_level_3_5dte, "GEX Level 3 (5D)", color=color.new(color.yellow, 30), linewidth=2)
plot(put_wall_5dte, "Put Wall (5D)", color=color.new(color.red, 30), linewidth=3)
plot(call_wall_5dte, "Call Wall (5D)", color=color.new(color.green, 30), linewidth=3)
'''

    print("📋 CÓDIGO PINE SCRIPT PARA TRADINGVIEW (5 DTE):")
    print(tradingview_code_5dte)
    print("\n✅ Copie o código acima e cole no TradingView Pine Editor!")
else:
    print("⚠️ Nenhum dado disponível para 5 DTE")



🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP (5 DTE - VERSÃO SIMPLIFICADA)



NameError: name 'dfAgg_5dte' is not defined

In [109]:
# ========================================
# CÉLULA 33 - GERADOR TRADINGVIEW - GAMMA (0 DTE)
# ========================================
# ==================== CÉLULA FINAL DO NOTEBOOK (0 DTE) ====================
print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP (0 DTE - VERSÃO SIMPLIFICADA)")
print("="*80 + "\n")

# ==================== COLETA DOS DADOS (0 DTE) ====================
if len(dfAgg_0dte) > 0:
    # CHART 1 - Spot Gamma Levels (0 DTE)
    dfAgg_sorted_0dte = dfAgg_0dte.sort_values(by='TotalGamma')
    smallest_gamma_0dte = dfAgg_sorted_0dte.head(3)
    largest_gamma_0dte = dfAgg_sorted_0dte.tail(3)

    put_wall_0dte = smallest_gamma_0dte.index[0] if len(smallest_gamma_0dte) > 0 else 0
    call_wall_0dte = largest_gamma_0dte.index[-1] if len(largest_gamma_0dte) > 0 else 0

    # CHART 2 - GEX Levels (0 DTE)
    dfAgg_0dte['AbsoluteTotalGEX'] = dfAgg_0dte['CallGEX'].abs() + dfAgg_0dte['PutGEX'].abs()
    dfAgg_sorted_gex_0dte = dfAgg_0dte.sort_values(by='AbsoluteTotalGEX', ascending=False)
    gex_levels_0dte = dfAgg_sorted_gex_0dte.head(6)

    gex_level_1_0dte = gex_levels_0dte.index[0] if len(gex_levels_0dte) > 0 else 0
    gex_level_2_0dte = gex_levels_0dte.index[1] if len(gex_levels_0dte) > 1 else 0
    gex_level_3_0dte = gex_levels_0dte.index[2] if len(gex_levels_0dte) > 2 else 0
    gex_level_4_0dte = gex_levels_0dte.index[3] if len(gex_levels_0dte) > 3 else 0
    gex_level_5_0dte = gex_levels_0dte.index[4] if len(gex_levels_0dte) > 4 else 0
    gex_level_6_0dte = gex_levels_0dte.index[5] if len(gex_levels_0dte) > 5 else 0

    # ==================== GERAR CÓDIGO TRADINGVIEW (0 DTE) ====================
    tradingview_code_0dte = f'''
// GEX Levels (0 DTE)
gex_level_1_0dte = {gex_level_1_0dte:.2f}
gex_level_2_0dte = {gex_level_2_0dte:.2f}
gex_level_3_0dte = {gex_level_3_0dte:.2f}
gex_level_4_0dte = {gex_level_4_0dte:.2f}
gex_level_5_0dte = {gex_level_5_0dte:.2f}
gex_level_6_0dte = {gex_level_6_0dte:.2f}

// Gamma Walls (0 DTE)
put_wall_0dte = {put_wall_0dte:.2f}
call_wall_0dte = {call_wall_0dte:.2f}

// Plot lines
plot(gex_level_1_0dte, "GEX Level 1 (0D)", color=color.new(color.yellow, 50), linewidth=1, style=plot.style_circles)
plot(gex_level_2_0dte, "GEX Level 2 (0D)", color=color.new(color.yellow, 50), linewidth=1, style=plot.style_circles)
plot(gex_level_3_0dte, "GEX Level 3 (0D)", color=color.new(color.yellow, 50), linewidth=1, style=plot.style_circles)
plot(put_wall_0dte, "Put Wall (0D)", color=color.new(color.red, 50), linewidth=2)
plot(call_wall_0dte, "Call Wall (0D)", color=color.new(color.green, 50), linewidth=2)
'''

    print("📋 CÓDIGO PINE SCRIPT PARA TRADINGVIEW (0 DTE):")
    print(tradingview_code_0dte)
    print("\n✅ Copie o código acima e cole no TradingView Pine Editor!")
else:
    print("⚠️ Nenhum dado disponível para 0 DTE")



🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP (0 DTE - VERSÃO SIMPLIFICADA)



NameError: name 'dfAgg_0dte' is not defined

### RESULTADOS DELTA

In [112]:
# ========================================
# CÉLULA 34 - MARKDOWN
# ========================================
# ### RESULTADOS DELTA

# ========================================
# CÉLULA 35 - GERADOR TRADINGVIEW - DELTA (ALL)
# ========================================
# ==================== CÉLULA FINAL DO NOTEBOOK - DELTA ====================
print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - DELTA")
print("="*80 + "\n")

# ==================== COLETA E CÁLCULO DOS DADOS - DELTA ====================
# CHART 4 - Delta Walls
smallest_delta = dfAgg_delta.sort_values(by='TotalDelta').head(3)
largest_delta = dfAgg_delta.sort_values(by='TotalDelta').tail(3)

put_delta_wall = smallest_delta.index[0]
call_delta_wall = largest_delta.index[2]

# ==================== GERAR CÓDIGO TRADINGVIEW - DELTA ====================
tradingview_code_delta = f'''
// Delta Walls
put_delta_wall = {put_delta_wall:.2f}
call_delta_wall = {call_delta_wall:.2f}

// Plot Delta lines
plot(put_delta_wall, "Put Delta Wall", color=color.purple, linewidth=3, style=plot.style_cross)
plot(call_delta_wall, "Call Delta Wall", color=color.blue, linewidth=3, style=plot.style_cross)
'''

print("📋 CÓDIGO PINE SCRIPT PARA TRADINGVIEW (DELTA):")
print(tradingview_code_delta)
print("\n✅ Copie o código acima e cole no TradingView Pine Editor!")




🚀 GERADOR DE CÓDIGO TRADINGVIEW - DELTA

📋 CÓDIGO PINE SCRIPT PARA TRADINGVIEW (DELTA):

// Delta Walls
put_delta_wall = 35500.00
call_delta_wall = 22250.00

// Plot Delta lines
plot(put_delta_wall, "Put Delta Wall", color=color.purple, linewidth=3, style=plot.style_cross)
plot(call_delta_wall, "Call Delta Wall", color=color.blue, linewidth=3, style=plot.style_cross)


✅ Copie o código acima e cole no TradingView Pine Editor!


In [111]:
# ========================================
# CÉLULA 36 - GERADOR TRADINGVIEW - DELTA (5 DTE)
# ========================================
# ==================== CÉLULA FINAL DO NOTEBOOK - DELTA (5 DTE) ====================
print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - DELTA (5 DTE)")
print("="*80 + "\n")

if len(dfAgg_5dte_delta) > 0:
    # CHART 4 - Delta Walls (5 DTE)
    smallest_delta_5dte = dfAgg_5dte_delta.sort_values(by='TotalDelta').head(3)
    largest_delta_5dte = dfAgg_5dte_delta.sort_values(by='TotalDelta').tail(3)

    put_delta_wall_5dte = smallest_delta_5dte.index[0] if len(smallest_delta_5dte) > 0 else 0
    call_delta_wall_5dte = largest_delta_5dte.index[-1] if len(largest_delta_5dte) > 0 else 0

    # ==================== GERAR CÓDIGO TRADINGVIEW - DELTA (5 DTE) ====================
    tradingview_code_delta_5dte = f'''
// Delta Walls (5 DTE)
put_delta_wall_5dte = {put_delta_wall_5dte:.2f}
call_delta_wall_5dte = {call_delta_wall_5dte:.2f}

// Plot Delta lines (5 DTE)
plot(put_delta_wall_5dte, "Put Delta Wall (5D)", color=color.new(color.purple, 30), linewidth=3, style=plot.style_cross)
plot(call_delta_wall_5dte, "Call Delta Wall (5D)", color=color.new(color.blue, 30), linewidth=3, style=plot.style_cross)
'''

    print("📋 CÓDIGO PINE SCRIPT PARA TRADINGVIEW (DELTA 5 DTE):")
    print(tradingview_code_delta_5dte)
    print("\n✅ Copie o código acima e cole no TradingView Pine Editor!")
else:
    print("⚠️ Nenhum dado disponível para Delta 5 DTE")



🚀 GERADOR DE CÓDIGO TRADINGVIEW - DELTA (5 DTE)



NameError: name 'dfAgg_5dte_delta' is not defined

In [113]:
# ========================================
# CÉLULA 37 - GERADOR TRADINGVIEW - DELTA (0 DTE)
# ========================================
# ==================== CÉLULA FINAL DO NOTEBOOK - DELTA (0 DTE) ====================
print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - DELTA (0 DTE)")
print("="*80 + "\n")

if len(dfAgg_0dte_delta) > 0:
    # CHART 4 - Delta Walls (0 DTE)
    smallest_delta_0dte = dfAgg_0dte_delta.sort_values(by='TotalDelta').head(3)
    largest_delta_0dte = dfAgg_0dte_delta.sort_values(by='TotalDelta').tail(3)

    put_delta_wall_0dte = smallest_delta_0dte.index[0] if len(smallest_delta_0dte) > 0 else 0
    call_delta_wall_0dte = largest_delta_0dte.index[-1] if len(largest_delta_0dte) > 0 else 0

    # ==================== GERAR CÓDIGO TRADINGVIEW - DELTA (0 DTE) ====================
    tradingview_code_delta_0dte = f'''
// Delta Walls (0 DTE)
put_delta_wall_0dte = {put_delta_wall_0dte:.2f}
call_delta_wall_0dte = {call_delta_wall_0dte:.2f}

// Plot Delta lines (0 DTE)
plot(put_delta_wall_0dte, "Put Delta Wall (0D)", color=color.new(color.purple, 50), linewidth=2, style=plot.style_cross)
plot(call_delta_wall_0dte, "Call Delta Wall (0D)", color=color.new(color.blue, 50), linewidth=2, style=plot.style_cross)
'''

    print("📋 CÓDIGO PINE SCRIPT PARA TRADINGVIEW (DELTA 0 DTE):")
    print(tradingview_code_delta_0dte)
    print("\n✅ Copie o código acima e cole no TradingView Pine Editor!")
else:
    print("⚠️ Nenhum dado disponível para Delta 0 DTE")


🚀 GERADOR DE CÓDIGO TRADINGVIEW - DELTA (0 DTE)



NameError: name 'dfAgg_0dte_delta' is not defined

### RESUMO FINAL

In [114]:
# ========================================
# CÉLULA 38 - RESUMO FINAL
# ========================================
print("\n" + "="*80)
print("📊 RESUMO COMPLETO DA ANÁLISE")
print("="*80 + "\n")

print("### GAMMA - GEX ###")
print(f"Total Gamma: ${df['TotalGamma'].sum():,.2f} Bn")
print(f"Call Gamma: ${dfAgg['CallGEX'].sum() / 10**9:,.2f} Bn")
print(f"Put Gamma: ${dfAgg['PutGEX'].sum() / 10**9:,.2f} Bn")
print(f"Put Wall: {put_wall:.2f}")
print(f"Call Wall: {call_wall:.2f}")
print(f"Gamma Flip: {gamma_flip:.2f}")

print("\n### DELTA - DEX ###")
print(f"Total Delta: ${df['TotalDelta'].sum():,.2f} Million")
print(f"Call Delta: ${dfAgg_delta['CallDEX'].sum() / 10**6:,.2f} Million")
print(f"Put Delta: ${dfAgg_delta['PutDEX'].sum() / 10**6:,.2f} Million")
print(f"Put Delta Wall: {put_delta_wall:.2f}")
print(f"Call Delta Wall: {call_delta_wall:.2f}")

print("\n" + "="*80)
print("✅ ANÁLISE COMPLETA FINALIZADA!")
print("="*80)


📊 RESUMO COMPLETO DA ANÁLISE

### GAMMA - GEX ###
Total Gamma: $-0.04 Bn
Call Gamma: $0.55 Bn
Put Gamma: $-0.60 Bn
Put Wall: 23300.00
Call Wall: 25900.00
Gamma Flip: 0.00

### DELTA - DEX ###
Total Delta: $34.37 Million
Call Delta: $122.30 Million
Put Delta: $-87.93 Million
Put Delta Wall: 35500.00
Call Delta Wall: 22250.00

✅ ANÁLISE COMPLETA FINALIZADA!
